In [ ]:
!pip install -q ultralytics gradio

import os
import time
import yaml
import torch
from ultralytics import YOLO

print("GPU Kullanılıyor mu?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Modeli:", torch.cuda.get_device_name(0))

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
GPU Kullanılıyor mu?: True
GPU Modeli: Tesla T4


In [ ]:
!yolo val model=yolov8n.pt data=VOC.yaml imgsz=640 batch=16 split=val > /dev/null 2>&1

voc_custom = {
    'path': '/content/datasets/VOC',
    'train': ['images/train2012', 'images/train2007'],
    'val': ['images/val2012', 'images/val2007'],
    'test': 'images/test2007',
    'names': {
        0: 'aeroplane', 1: 'bicycle', 2: 'bird', 3: 'boat', 4: 'bottle',
        5: 'bus', 6: 'car', 7: 'cat', 8: 'chair', 9: 'cow',
        10: 'diningtable', 11: 'dog', 12: 'horse', 13: 'motorbike', 14: 'person',
        15: 'pottedplant', 16: 'sheep', 17: 'sofa', 18: 'train', 19: 'tvmonitor'
    }
}

with open('voc_3way.yaml', 'w') as f:
    yaml.dump(voc_custom, f, sort_keys=False)

print("voc_3way.yaml oluşturuldu.")

voc_3way.yaml oluşturuldu.


In [ ]:
model = YOLO('yolov8n.pt')

start_train = time.time()

results = model.train(
    data='voc_3way.yaml',
    epochs=20,
    imgsz=640,
    batch=16,
    optimizer='AdamW',
    lr0=0.001,
    project='cv_odevi',
    name='yolov8n_voc'
)

train_time_sec = time.time() - start_train
print(f"Toplam Eğitim Süresi: {train_time_sec/60:.2f} dakika ({train_time_sec:.1f} saniye)")

Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=voc_3way.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_voc, nbs=64, nms=None, opset=None, optimize=False, optimizer=AdamW

In [ ]:
import glob
import numpy as np
from ultralytics.utils.metrics import box_iou

best_model = YOLO('/content/runs/detect/cv_odevi/yolov8n_voc/weights/best.pt')

metrics = best_model.val(data='voc_3way.yaml', split='test', imgsz=640)
print(f"Test mAP@0.5:      {metrics.box.map50:.4f}")
print(f"Test mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Test Precision:    {metrics.box.mp:.4f}")
print(f"Test Recall:       {metrics.box.mr:.4f}")

test_imgs = sorted(glob.glob('/content/datasets/VOC/images/test2007/*.jpg'))[:100]
iou_list = []
times = []

_ = best_model.predict(test_imgs[0], verbose=False)

for img_path in test_imgs:
    t0 = time.time()
    pred = best_model.predict(img_path, conf=0.25, verbose=False)[0]
    times.append((time.time() - t0) * 1000)

    lbl_path = img_path.replace('/images/', '/labels/').replace('.jpg', '.txt')
    if os.path.exists(lbl_path) and len(pred.boxes) > 0:
        h, w = pred.orig_shape
        gt_boxes = []
        with open(lbl_path) as f:
            for line in f:
                _, xc, yc, bw, bh = map(float, line.strip().split())
                gt_boxes.append([(xc - bw/2)*w, (yc - bh/2)*h, (xc + bw/2)*w, (yc + bh/2)*h])
        if gt_boxes:
            gt_t = torch.tensor(gt_boxes, dtype=torch.float32)
            pr_t = pred.boxes.xyxy.cpu()
            ious = box_iou(gt_t, pr_t)
            max_ious = ious.max(dim=1).values
            iou_list.extend(max_ious[max_ious > 0].tolist())

print(f"Ortalama IoU: {np.mean(iou_list):.4f}")
print(f"Ortalama Tahmin Süresi (Inference Time): {np.mean(times):.2f} ms / görsel")

Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,137,324 parameters, 0 gradients, 8.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 64.9±98.2 MB/s, size: 94.4 KB)
val: Scanning /content/datasets/VOC/labels/test2007.cache... 4952 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4952/4952 340.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 310/310 6.0it/s 51.4s
                   all       4952      12032      0.763      0.711      0.778      0.562
             aeroplane        204        285      0.843      0.753      0.839        0.6
               bicycle        239        337       0.85      0.788      0.872       0.64
                  bird        282        459      0.815      0.596      0.745        0.5
                  boat        172        263      0.691      0.612      0.687      0.426
                bottle        212   

In [ ]:
import time
import gradio as gr
from ultralytics import YOLO

model = YOLO('/content/runs/detect/cv_odevi/yolov8n_voc/weights/best.pt')

def detect_objects(image, conf_threshold):
    if image is None:
        return None, "Lütfen bir görsel yükleyin."

    t0 = time.time()
    results = model.predict(image, conf=conf_threshold, imgsz=640, verbose=False)[0]
    inf_ms = (time.time() - t0) * 1000

    annotated_img = results.plot()[:, :, ::-1]

    detections = []
    for box in results.boxes:
        cls_name = model.names[int(box.cls[0])]
        conf = float(box.conf[0])
        detections.append(f"- {cls_name}: %{conf*100:.1f}")

    summary = f"Tahmin Süresi: {inf_ms:.1f} ms\n\nTespit Edilenler:\n"
    summary += "\n".join(detections) if detections else "Nesne tespit edilemedi."
    return annotated_img, summary

demo = gr.Interface(
    fn=detect_objects,
    inputs=[
        gr.Image(type="numpy", label="Fotoğraf Yükle"),
        gr.Slider(0.1, 0.9, value=0.25, step=0.05, label="Güven Eşiği (Confidence)")
    ],
    outputs=[
        gr.Image(type="numpy", label="Model Çıktısı"),
        gr.Textbox(label="Tespit Özeti ve Süre")
    ],
    title="Pascal VOC Nesne Tespiti (YOLOv8n)",
    description="Bir fotoğraf yükleyin; model 20 Pascal VOC sınıfı içinde nesne tespiti yapsın."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3ca71becff0d9f4d98.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
